# 03 - Modelagem Dimensional Ouro: Vendas/Pedidos

Lê a camada **Prata** (`lakehouse.prata.data_api`), aplica modelagem
dimensional (esquema estrela) para análise de **vendas/pedidos** e grava
as tabelas resultantes em Iceberg na camada **Ouro**:

- `lakehouse.ouro.tab_dim_produto`     -> `s3a://lakehouse/ouro/tab_dim_vendas`
- `lakehouse.ouro.tab_dim_categoria`   -> `s3a://lakehouse/ouro/tab_dim_vendas`
- `lakehouse.ouro.tab_dim_data`        -> `s3a://lakehouse/ouro/tab_dim_vendas`
- `lakehouse.ouro.tab_fato_vendas`     -> `s3a://lakehouse/ouro/tab_fato_vendas`

> Observação: a Fake Store API não expõe um endpoint de pedidos/vendas
> realista e estável. Para fins didáticos, **simulamos pedidos** a partir
> dos produtos da camada Prata (cada produto gera N pedidos simulados
> com quantidade e data aleatórias). Em um cenário real, a camada Bronze
> teria sido alimentada por uma API/DB de pedidos de fato.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, monotonically_increasing_id, lit, expr, year, month, dayofmonth,
    quarter, dayofweek, date_format, current_timestamp
)
import random
from datetime import datetime, timedelta

## 1. Configuração da SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("ouro-modelagem-vendas")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.postgresql:postgresql:42.7.4")

    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "jdbc")
    .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://postgres:5432/metastore")
    .config("spark.sql.catalog.lakehouse.jdbc.user", "dba_admin")
    .config("spark.sql.catalog.lakehouse.jdbc.password", "Agent9-Backtalk6-Zestfully5-Luxury1-Calculus2")
    .config("spark.sql.catalog.lakehouse.jdbc.driver", "org.postgresql.Driver")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin_minio")
    .config("spark.hadoop.fs.s3a.secret.key", "Grunt9-Relenting6-Hula5-Catcall9-Residue3")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 2. Leitura da camada Prata

In [ ]:
df_prata = spark.table("lakehouse.prata.data_api")
df_prata.show(5, truncate=30)
print(f"Total de produtos na Prata: {df_prata.count()}")

## 3. Criação do schema Ouro

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS lakehouse.ouro")

## 4. Dimensão Produto — `tab_dim_produto`

Atributos descritivos do produto, sem métricas.

In [ ]:
df_dim_produto = (
    df_prata
    .select(
        col("id").alias("sk_produto"),
        col("title").alias("nome_produto"),
        col("category").alias("categoria"),
        col("price").alias("preco_unitario"),
        col("rating_rate").alias("avaliacao_media"),
        col("rating_count").alias("qtd_avaliacoes"),
    )
    .dropDuplicates(["sk_produto"])
)

df_dim_produto.show(5, truncate=30)

## 5. Dimensão Categoria — `tab_dim_categoria`

Lista única de categorias com chave substituta (`sk_categoria`).

In [ ]:
df_dim_categoria = (
    df_prata
    .select("category")
    .distinct()
    .withColumnRenamed("category", "categoria")
    .withColumn("sk_categoria", monotonically_increasing_id())
    .select("sk_categoria", "categoria")
)

df_dim_categoria.show()

## 6. Dimensão Data — `tab_dim_data`

Calendário cobrindo os últimos 365 dias até hoje, com atributos derivados
(ano, mês, dia, trimestre, dia da semana).

In [ ]:
hoje = datetime.today().date()
datas = [hoje - timedelta(days=i) for i in range(365)]

df_dim_data = spark.createDataFrame(
    [(d,) for d in datas], schema=["data"]
)

df_dim_data = (
    df_dim_data
    .withColumn("sk_data", date_format(col("data"), "yyyyMMdd").cast("int"))
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
    .withColumn("dia", dayofmonth(col("data")))
    .withColumn("trimestre", quarter(col("data")))
    .withColumn("dia_semana", dayofweek(col("data")))
    .withColumn("nome_mes", date_format(col("data"), "MMMM"))
    .select("sk_data", "data", "ano", "mes", "dia", "trimestre", "dia_semana", "nome_mes")
)

df_dim_data.show(5)

## 7. Fato Vendas — `tab_fato_vendas`

**Simulação**: cada produto gera entre 1 e 5 "pedidos" com quantidade
e data aleatórias dentro dos últimos 365 dias. `valor_total` = `quantidade * preco_unitario`.

> Substitua esta célula pela leitura de uma fonte real de pedidos quando disponível
> (ex: tabela `lakehouse.bronze.pedidos_api` alimentada por uma API de e-commerce real).

In [ ]:
random.seed(42)

produtos_pdf = df_dim_produto.select("sk_produto", "preco_unitario").toPandas()
datas_pdf = df_dim_data.select("sk_data").toPandas()["sk_data"].tolist()

linhas_fato = []
pedido_id = 1
for _, row in produtos_pdf.iterrows():
    n_pedidos = random.randint(1, 5)
    for _ in range(n_pedidos):
        quantidade = random.randint(1, 10)
        sk_data = random.choice(datas_pdf)
        linhas_fato.append({
            "sk_pedido": pedido_id,
            "sk_produto": int(row["sk_produto"]),
            "sk_data": int(sk_data),
            "quantidade": quantidade,
            "preco_unitario": float(row["preco_unitario"]),
            "valor_total": round(quantidade * float(row["preco_unitario"]), 2),
        })
        pedido_id += 1

import pandas as pd
fato_pdf = pd.DataFrame(linhas_fato)

df_fato_vendas = spark.createDataFrame(fato_pdf)
df_fato_vendas = df_fato_vendas.withColumn("data_processamento", current_timestamp())

df_fato_vendas.show(10)
print(f"Total de registros no fato: {df_fato_vendas.count()}")

## 8. Criação das tabelas Iceberg na camada Ouro

Tabelas dimensão -> `s3a://lakehouse/ouro/tab_dim_vendas`
(cada dimensão em sua própria subpasta de tabela)
Tabela fato -> `s3a://lakehouse/ouro/tab_fato_vendas`

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.ouro.tab_dim_produto (
    sk_produto      INT,
    nome_produto    STRING,
    categoria       STRING,
    preco_unitario  DOUBLE,
    avaliacao_media DOUBLE,
    qtd_avaliacoes  INT
)
USING iceberg
LOCATION 's3a://lakehouse/ouro/tab_dim_vendas/tab_dim_produto'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.ouro.tab_dim_categoria (
    sk_categoria BIGINT,
    categoria    STRING
)
USING iceberg
LOCATION 's3a://lakehouse/ouro/tab_dim_vendas/tab_dim_categoria'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.ouro.tab_dim_data (
    sk_data    INT,
    data       DATE,
    ano        INT,
    mes        INT,
    dia        INT,
    trimestre  INT,
    dia_semana INT,
    nome_mes   STRING
)
USING iceberg
LOCATION 's3a://lakehouse/ouro/tab_dim_vendas/tab_dim_data'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.ouro.tab_fato_vendas (
    sk_pedido          BIGINT,
    sk_produto         INT,
    sk_data            INT,
    quantidade         INT,
    preco_unitario     DOUBLE,
    valor_total        DOUBLE,
    data_processamento TIMESTAMP
)
USING iceberg
LOCATION 's3a://lakehouse/ouro/tab_fato_vendas'
""")

## 9. Carga das tabelas (overwrite — dimensões e fato recalculados a cada execução)

Diferente da Bronze/Prata (incremental via MERGE), aqui usamos `overwrite`
pois as dimensões/fato são derivadas e recalculadas a partir da Prata
a cada execução do pipeline.

In [ ]:
df_dim_produto.writeTo("lakehouse.ouro.tab_dim_produto").overwritePartitions()
df_dim_categoria.writeTo("lakehouse.ouro.tab_dim_categoria").overwritePartitions()
df_dim_data.writeTo("lakehouse.ouro.tab_dim_data").overwritePartitions()
df_fato_vendas.writeTo("lakehouse.ouro.tab_fato_vendas").overwritePartitions()

## 10. Verificação

In [ ]:
spark.sql("SELECT * FROM lakehouse.ouro.tab_dim_produto LIMIT 5").show(truncate=30)
spark.sql("SELECT * FROM lakehouse.ouro.tab_dim_categoria").show()
spark.sql("SELECT * FROM lakehouse.ouro.tab_dim_data LIMIT 5").show()
spark.sql("SELECT * FROM lakehouse.ouro.tab_fato_vendas LIMIT 10").show()

In [ ]:
spark.sql("""
SELECT
    p.categoria,
    SUM(f.valor_total) AS receita_total,
    SUM(f.quantidade)  AS qtd_total
FROM lakehouse.ouro.tab_fato_vendas f
JOIN lakehouse.ouro.tab_dim_produto p ON f.sk_produto = p.sk_produto
GROUP BY p.categoria
ORDER BY receita_total DESC
""").show()

In [ ]:
spark.stop()